# Análise exploratória dos anúncios do Airbnb em Santiago, Chile

**Cidade escolhida:** Santiago, Região Metropolitana de Santiago, Chile

(n tenho ideia doq mais fazer sem ser super óbvio e a pertuna que pegeui ´ta uma bosta)
**Perguntas de análise**
1. Como o preço de casas/apartamentos inteiros varia entre anúncios com e sem aquecimento e ar-condicionado?
2.
3.

## Carregamento e Estatísticas Básicas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats # will be used for the statistic test

sns.set_theme(style="whitegrid")

# listings from Santiago, Región Metropolitana de Santiago, Chile
datapath = "data/listings.csv"

df = pd.read_csv(datapath)  
print(df.info())

A célula acima carrega o conjunto de dados em um `DataFrame` e imprime alguns atributos sobre as colunas e os valores presentes. O conjunto de dados contém 18534 registros e 90 colunas. É possível notar que algumas colunas, como `host_since`, `neighbourhood`, e `instant_bookable`, estão completamente vazias. 

Abaixo, são identificadas as colunas com maior proporção de valores ausentes.

In [ ]:
cols = df.columns
cols_nulls = df[cols].isnull()

cols_stats = pd.DataFrame({
    "qtd_ausentes": (cols_nulls.sum()),
    "%_ausentes": (cols_nulls.mean() * 100),
}).sort_values("%_ausentes")

cols_stats[cols_stats["qtd_ausentes"] > 0]

É possível notar que a coluna `price` possui valores ausentes. Esses registros serão desconsideranos das análises que dependem do preço.

## Transformações e Variáveis Derivadas

A coluna `price` está em formato de texto (como `"$43,332.00"`). Foram removidos os caracteres `$` e `,` e então convertido o restante para o tipo `float`.

Variáveis derivadas: `price_accomodate`

In [ ]:
price = "price"
accommodates = "accommodates"
price_accomodate = "price_accomodate"

# remove '$' and ',' from `price` and convert the remainder to float
df[price] = df[price].astype("string").str.replace(r"[$,]", "", regex=True).astype(float)

# other proccessings needed


# derived variables
df[price_accomodate] = df[price] / df[accommodates]

df[[price, accommodates, price_accomodate]]

In [ ]:
print(f"{price} - Tendência Central e Medidas de Dispersão:")
df[price].describe()

In [ ]:
print(f"{price_accomodate} - Tendência Central e Medidas de Dispersão:")
df[price_accomodate].describe()

In [ ]:
prices = df[price].dropna()
mean, median, skew, p99 = prices.mean(), prices.median(), prices.skew(), prices.quantile(0.99)

skew_stats = pd.DataFrame({
    "media": mean,
    "mediana": median,
    "assimetria": skew,
    "p99": p99,
    "max": [df[price].max()],
}, index = [price])

print("Assimetria:")
skew_stats

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(prices[prices <= p99], bins = 50, ax = ax)

ax.axvline(mean, color="red", linestyle="--", label=f"Média: {mean:,.0f}")
ax.axvline(median, color="black", linestyle="-", label=f"Mediana: {median:,.0f}")

ax.set_title("Distribuição do preço por noite (até o percentil 99)")
ax.set_xlabel("Preço")
ax.set_ylabel("Número de anúncios")
ax.legend()

plt.show()

Os preços são assimétricos à direita: a média (~118 mil CLP) é cerca de o dobro da mediana (59 mil CLP); a assimetria (`skew`) é positiva; e o valor máximo é aproximadamente 141 vezes o percentil 99, o que indica uma cauda longa de anúncios muito caros. 

Como a média é sensível a esses valores extremos e a mediana não, foi dada preferência à mediana para a construção de tabelas e visualizações.

### Amenidades como variáveis binárias

A coluna `amenities` é uma string JSON que representa uma lista de amenidades. Foram criadas colunas binárias representando as amenidades utilizadas nas visualizações.

**Amenidades**

| Coluna | Justificativa |
|---|---|
| `am_air_conditioner` | Santiago tem verões quentes e secos; é um diferencial de conforto. |
| `am_heating` | Os invernos são frios e muitas casas não têm calefação central. |


In [ ]:

am_air_conditioner = "am_air_conditioner"
am_heating = "am_heating"

# find amenities that match with the terms used in the dataset
df[am_heating] = df["amenities"].str.contains(r"heating|heater|heat pump", case = False, na = False)
df[am_air_conditioner] = df["amenities"].str.contains(r"air condition|a/c", case = False, na = False)

group_temp = "group_temp"

# sets `group_temp` according to the amenities present
df[group_temp] = "Nenhum"
df.loc[df[am_heating] & ~df[am_air_conditioner], group_temp] = "Aquecimento"
df.loc[~df[am_heating] & df[am_air_conditioner], group_temp] = "Ar-condicionado"
df.loc[df[am_heating] & df[am_air_conditioner], group_temp] = "Ambos"

print(df[group_temp].value_counts())
print(df[group_temp].value_counts().sum() == len(df))

## Visualizações

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

sns.set_theme(style="whitegrid")

### Questão 1

**Filtros aplicados**

- Apenas anúncios com `room_type == "Entire home/apt"` (imóvel inteiro) foram considerados. Quartos compartilhados são um produto diferente e têm preços menores, o que poderia distorcer a comparação entre grupos de amenidades.

- Apenas anúncios com `price` não-nulo foram considerados.

**Grupos:** cada anúncio é classificado em um de quatro grupos, de acordo com as amenidades presentes: *Nenhum*, *Aquecimento*, *Ar-condicionado*, ou *Ambos*. Foram considerados aquecimento os itens "Heating", "Central heating", "Radiant heating", "Portable heater", "Heat pump" e "Heating - split type ductless system"; psicinas aquecidas e outros artigos de luxo foram desconsiderados.

**Agregação:** para cada grupo, `n` é o número de anúncios, `mediana` é a mediana do preço por noite, e `q1`/`q3` são o 1º e o 3º quartis. Usamos a mediana porque os preços são assimétricos à direita.

In [ ]:
classes = ["Nenhum", "Aquecimento", "Ar-condicionado", "Ambos"]
entire_homes = df[(df["room_type"] == "Entire home/apt") & df[price].notna()]

price_group = entire_homes.groupby(group_temp)[price]

tab = pd.DataFrame({
    "n": price_group.count(),
    "mediana": price_group.median(),
    "q1": price_group.quantile(0.25),
    "q3": price_group.quantile(0.75),
})

tab = tab.reindex(classes)
tab

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# needs to remove outliers ("fliers") since there are quite a lot of luxury houses 
sns.boxplot(data = entire_homes, x = group_temp, y = price, order = classes, showfliers = False, ax = ax)

# maybe log scale?
# ax.set_yscale("log")
ax.set_xticks(range(len(classes)))
ax.set_xticklabels([f"{c}\n(n = {tab.loc[c, 'n']})" for c in classes])
ax.set_title("Preço por noite conforme aquecimento e ar-condicionado\n(casas/apartamentos")
ax.set_xlabel("Grupo de Amenidades")
ax.set_ylabel("Preço por noite")

plt.show()